# Network Traffic Attribution under Spoofing & Decoy Attacks — Colab Demo

Self-contained Colab notebook. Run the cells top to bottom — no manual upload required.

1. Clone the repo and install dependencies.
2. Load the bundled 10 000-row attribution sample.
3. Quick EDA (class balance, top discriminative features).
4. Pick a scenario:
   - **A — Use pretrained models** (~5 s): loads `rf.pkl`, `xgb.pkl`, `mlp.pkl` and scores the held-out test split.
   - **B — Train from scratch** (~30 s): fits Random Forest + XGBoost + MLP with 5-fold CV and re-scores.
5. Plots: confusion matrices, ROC curves, feature importance, metric comparison.

## 1. Setup — upload the project and install dependencies

**In Google Colab:** the runtime starts empty, so it needs the project files before the rest of the notebook can run. The cell below will:

1. Detect whether the project is already on disk (i.e. you're running locally from a checkout, or a previous run already uploaded it).
2. If not, open Colab's file picker so you can upload either:
   - a single zip of the project folder (recommended — fastest), or
   - the individual data / model files needed by the notebook.
3. Install the Python dependencies (`xgboost`, `seaborn`, `joblib`).

If you're running this locally, the upload step is skipped automatically.

In [ ]:
# --- Colab setup -------------------------------------------------------
# Make sure the project files are available, then install deps.
import os, sys, zipfile

IN_COLAB = 'google.colab' in sys.modules
PROJECT  = 'Network-Traffic-Attribution'
ANCHOR   = 'data/sample/attribution_sample.csv'   # we check for this to know we're inside the project root

def _try_locate_project_dir():
    """If we landed next to the project folder, cd into it."""
    if os.path.isfile(ANCHOR):
        return True                                # already at root
    for name in os.listdir('.'):
        if os.path.isdir(name) and os.path.isfile(os.path.join(name, ANCHOR)):
            os.chdir(name)
            print(f'cd into {name}/')
            return True
    return False

if not _try_locate_project_dir():
    if IN_COLAB:
        print('Upload the project zip (or its individual files) — the picker will appear:')
        from google.colab import files
        uploaded = files.upload()
        # Unzip any zip the user uploaded.
        for fname in list(uploaded):
            if fname.lower().endswith('.zip'):
                with zipfile.ZipFile(fname) as z:
                    z.extractall('.')
                print(f'extracted {fname}')
        # Look for the project root again now that things may have been extracted.
        if not _try_locate_project_dir():
            raise RuntimeError(
                'Could not find data/sample/attribution_sample.csv after upload. '
                'Upload either a zip that contains the project folder, or at minimum '
                'the data/sample/ and models/ folders with their files.'
            )
    else:
        raise RuntimeError(
            f'Run this notebook from the project root, or from a folder that contains {PROJECT}/.'
        )

print('project root:', os.getcwd())
!pip install -q xgboost seaborn joblib

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, classification_report,
)
from xgboost import XGBClassifier
ROOT = Path('.').resolve()
print('working dir:', ROOT)

## 2. Load the bundled attribution sample

10 000 balanced flows (5 000 real attacker + 5 000 spoof/decoy) with 14 attribution features.

In [ ]:
df = pd.read_csv('data/sample/attribution_sample.csv')
print(f'rows: {len(df):,}')
print(f'real (1): {(df["label"]==1).sum():,}  |  spoof/decoy (0): {(df["label"]==0).sum():,}')
df.head()

## 3. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['label'].map({0:'spoof/decoy', 1:'real'}).value_counts().plot.bar(
    ax=axes[0], color=['#4C4C4C','#1A1A1A'])
axes[0].set_title('Class balance'); axes[0].tick_params(axis='x', rotation=0)
for feat, ax in zip(['ttl_variance', 'src_ip_entropy'], axes[1:]):
    for lab, color in [(0, '#888888'), (1, '#1A1A1A')]:
        ax.hist(df.loc[df['label']==lab, feat], bins=40, alpha=0.6,
                label=('spoof/decoy' if lab==0 else 'real'), color=color)
    ax.set_title(feat); ax.legend()
plt.tight_layout(); plt.show()

## 4. Prepare train/test split

In [ ]:
X = df.drop(columns=['label']).values
y = df['label'].values
feature_names = list(df.drop(columns=['label']).columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f'train: {len(X_train):,}   test: {len(X_test):,}')

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train); X_test_s = scaler.transform(X_test)

## 5. Scenario A — Use pretrained models

In [ ]:
# Score test split with the three pretrained .pkl files shipped in the repo.
rf = joblib.load('models/rf.pkl')
xgb = joblib.load('models/xgb.pkl')
mlp_bundle = joblib.load('models/mlp.pkl')
mlp, mlp_scaler = mlp_bundle['model'], mlp_bundle['scaler']

def metrics(y_true, y_pred, y_prob):
    return dict(accuracy=accuracy_score(y_true, y_pred),
                precision=precision_score(y_true, y_pred),
                recall=recall_score(y_true, y_pred),
                f1=f1_score(y_true, y_pred),
                roc_auc=roc_auc_score(y_true, y_prob))

rows = {}
for name, model, X_eval in [
    ('Random Forest', rf, X_test),
    ('XGBoost', xgb, X_test),
    ('MLP', mlp, mlp_scaler.transform(X_test)),
]:
    yp = model.predict(X_eval); pp = model.predict_proba(X_eval)[:, 1]
    rows[name] = metrics(y_test, yp, pp)

comparison = pd.DataFrame(rows).T.round(4)
comparison

## 6. Scenario B — Train from scratch

Skip this if you ran Scenario A. Otherwise this fits new RF + XGB + MLP.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight='balanced')
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    eval_metric='logloss', tree_method='hist', n_jobs=-1, random_state=42)
mlp = MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=200,
                    batch_size=256, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model, X_in in [
    ('Random Forest', rf, X_train),
    ('XGBoost', xgb, X_train),
    ('MLP', mlp, X_train_s),
]:
    s = cross_val_score(model, X_in, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f'{name:14s}  5-fold F1: {s.mean():.4f}  (+/- {s.std():.4f})')

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
mlp.fit(X_train_s, y_train)
print('Models trained.')

## 7. Confusion matrices + ROC + Feature importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, model, X_eval) in zip(axes, [
    ('Random Forest', rf, X_test),
    ('XGBoost', xgb, X_test),
    ('MLP', mlp, X_test_s),
]):
    yp = model.predict(X_eval)
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm, display_labels=['spoof/decoy', 'real']).plot(
        ax=ax, colorbar=False, cmap='Greys')
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for name, model, X_eval, style in [
    ('Random Forest', rf, X_test, '-'),
    ('XGBoost', xgb, X_test, '--'),
    ('MLP', mlp, X_test_s, ':'),
]:
    pp = model.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, pp)
    auc = roc_auc_score(y_test, pp)
    plt.plot(fpr, tpr, style, label=f'{name} (AUC={auc:.3f})', color='black')
plt.plot([0, 1], [0, 1], 'k-', alpha=0.3)
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC — Attribution models')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
plt.figure(figsize=(7, 5))
imp.plot.barh(color='#1A1A1A')
plt.xlabel('Importance'); plt.title('Feature importance — Random Forest')
plt.tight_layout(); plt.show()
print('\nTop 5 features:')
print(imp.sort_values(ascending=False).head(5).round(4))

## 8. Live inference demo

In [ ]:
# Score a handful of fresh flows.
samples = df.drop(columns=['label']).sample(5, random_state=7)
true_labels = df.loc[samples.index, 'label']
preds = rf.predict(samples.values)
probs = rf.predict_proba(samples.values)[:, 1]
out = samples.copy()
out['true']      = true_labels.map({0:'spoof/decoy', 1:'real'}).values
out['predicted'] = pd.Series(preds, index=samples.index).map({0:'spoof/decoy', 1:'real'}).values
out['prob_real'] = probs
out[['true','predicted','prob_real']]